# 10_silver_quality_audit.ipynb — Auditoría global de la capa Silver

Este notebook valida la capa **Silver** completa de DeepWave Canarias antes de pasar a Gold.

Objetivos:

```text
1. Inventariar todas las tablas Silver generadas.
2. Comprobar existencia, filas, columnas mínimas y muestras.
3. Consolidar todos los reportes quality_*.csv.
4. Validar fuentes reales, interpoladas y proxy/sintéticas.
5. Detectar errores críticos antes de crear Gold.
6. Generar un informe final de estado Silver.
```

Salidas principales:

```text
silver/_quality_reports/silver_global_audit_summary.csv
silver/_quality_reports/silver_sources_status.csv
silver/_quality_reports/silver_table_inventory.csv
silver/_quality_reports/silver_schema_audit.csv
silver/_quality_reports/silver_known_caveats.csv
silver/_metadata/dim_source_quality.csv
```

Interpretación:

```text
PASS = correcto
WARN = aceptable con salvedad documentada
FAIL = problema que debería corregirse antes de Gold
```

## Celda 0 — Montar Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Celda 1 — Instalar librerías necesarias

In [2]:
!pip -q install pyarrow pandas numpy tqdm

## Celda 2 — Imports, rutas y configuración

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import json
import re
import shutil
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/AI Projects/DeepWave Canarias")
SILVER_DIR = BASE_DIR / "silver"

QC_DIR = SILVER_DIR / "_quality_reports"
META_DIR = SILVER_DIR / "_metadata"

QC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

FAIL_ON_CRITICAL = True

# Para no leer tablas gigantes enteras por accidente.
MAX_FULL_READ_ROWS = 1_000_000

print("BASE_DIR:", BASE_DIR)
print("SILVER_DIR:", SILVER_DIR)
print("Existe SILVER_DIR:", SILVER_DIR.exists())

if not SILVER_DIR.exists():
    raise FileNotFoundError("No existe la carpeta silver/. Ejecuta primero los notebooks Silver.")

BASE_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias
SILVER_DIR: /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver
Existe SILVER_DIR: True


## Celda 3 — Definición oficial de tablas/fuentes esperadas

In [4]:
EXPECTED_OUTPUTS = [
    {
        "table": "beach_geography",
        "source": "DIM_ZONE",
        "path": SILVER_DIR / "beach_geography" / "beach_geography.parquet",
        "kind": "single_parquet",
        "required": True,
        "min_rows": 1,
        "expected_role": "static_dimension",
        "notes": "Dimensión geográfica base. Una fila por zona/playa.",
    },
    {
        "table": "ocean_hourly",
        "source": "SIMAR",
        "path": SILVER_DIR / "ocean_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_model_wave",
        "notes": "Oleaje histórico/modelado de SIMAR.",
    },
    {
        "table": "ocean_hourly",
        "source": "COPERNICUS_MARINE_WAVES",
        "path": SILVER_DIR / "ocean_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_model_wave_interpolated",
        "notes": "Oleaje Copernicus 3h interpolado a 1h con flag=3.",
    },
    {
        "table": "meteo_hourly",
        "source": "SIMAR",
        "path": SILVER_DIR / "meteo_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_model_wind",
        "notes": "Viento SIMAR.",
    },
    {
        "table": "meteo_hourly",
        "source": "ERA5",
        "path": SILVER_DIR / "meteo_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_reanalysis_precipitation",
        "notes": "ERA5 real disponible principalmente como precipitación/tp.",
    },
    {
        "table": "meteo_hourly",
        "source": "ERA5_PROXY",
        "path": SILVER_DIR / "meteo_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "proxy_meteorology",
        "notes": "Variables meteorológicas sintéticas/proxy. No usar como verdad de validación.",
    },
    {
        "table": "meteo_hourly",
        "source": "AEMET",
        "path": SILVER_DIR / "meteo_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "observed_daily_meteorology",
        "notes": "Datos diarios AEMET guardados en tabla común con temporal_resolution=daily.",
    },
    {
        "table": "tide_hourly",
        "source": "REDMAR",
        "path": SILVER_DIR / "tide_hourly",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "observed_sea_level",
        "notes": "Nivel del mar y mareas observadas REDMAR.",
    },
    {
        "table": "ocean_physics",
        "source": "SIMAR",
        "path": SILVER_DIR / "ocean_physics",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_model_ocean_physics",
        "notes": "Corrientes, temperatura y salinidad desde SIMAR.",
    },
    {
        "table": "ocean_physics",
        "source": "COPERNICUS_MARINE_PHYSICS",
        "path": SILVER_DIR / "ocean_physics",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "historical_model_ocean_physics_partial",
        "notes": "Copernicus physics parcial: temperatura/salinidad disponibles; uo/vo no disponibles en descarga.",
    },
    {
        "table": "ocean_validation",
        "source": "REDEXT_REDCOS",
        "path": SILVER_DIR / "ocean_validation",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "observed_wave_validation",
        "notes": "Boyas observadas para validación. Sin coordenadas en CSV: zona_id=CAN_VALIDATION_UNASSIGNED.",
    },
    {
        "table": "bathymetry_features",
        "source": "GEBCO_OR_EMODNET",
        "path": SILVER_DIR / "bathymetry_features" / "bathymetry_features.parquet",
        "kind": "single_parquet",
        "required": True,
        "min_rows": 1,
        "expected_role": "static_bathymetry_features",
        "notes": "Features batimétricas estáticas por zona.",
    },
    {
        "table": "forecast_gfs",
        "source": "NOAA_GFS",
        "path": SILVER_DIR / "forecast_gfs",
        "kind": "partitioned_source",
        "required": True,
        "min_rows": 1,
        "expected_role": "operational_forecast_with_proxy",
        "notes": "Forecast GFS/GFS-Wave. Incluye proxy sintético trazado para nulos costeros, swell y precipitación.",
    },
]

expected_outputs_df = pd.DataFrame(EXPECTED_OUTPUTS)
display(expected_outputs_df)

,table,source,path,kind,required,min_rows,expected_role,notes
0,beach_geography,DIM_ZONE,/content/drive/MyDrive/AI Projects/DeepWave Ca...,single_parquet,True,1,static_dimension,Dimensión geográfica base. Una fila por zona/p...
1,ocean_hourly,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_model_wave,Oleaje histórico/modelado de SIMAR.
2,ocean_hourly,COPERNICUS_MARINE_WAVES,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_model_wave_interpolated,Oleaje Copernicus 3h interpolado a 1h con flag=3.
3,meteo_hourly,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_model_wind,Viento SIMAR.
4,meteo_hourly,ERA5,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_reanalysis_precipitation,ERA5 real disponible principalmente como preci...
5,meteo_hourly,ERA5_PROXY,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,proxy_meteorology,Variables meteorológicas sintéticas/proxy. No ...
6,meteo_hourly,AEMET,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,observed_daily_meteorology,Datos diarios AEMET guardados en tabla común c...
7,tide_hourly,REDMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,observed_sea_level,Nivel del mar y mareas observadas REDMAR.
8,ocean_physics,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_model_ocean_physics,"Corrientes, temperatura y salinidad desde SIMAR."
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,/content/drive/MyDrive/AI Projects/DeepWave Ca...,partitioned_source,True,1,historical_model_ocean_physics_partial,Copernicus physics parcial: temperatura/salini...


## Celda 4 — Esquemas mínimos esperados

In [5]:
REQUIRED_COLUMNS = {
    "beach_geography": [
        "zona_id", "nombre_zona", "isla", "municipio", "lat", "lon",
        "tipo_zona", "orientacion_costa",
    ],
    "ocean_hourly": [
        "timestamp", "zona_id", "lat", "lon", "source",
        "hs", "tp", "wave_direction",
    ],
    "meteo_hourly": [
        "timestamp", "zona_id", "lat", "lon", "source",
        "wind_speed", "wind_direction", "temperature_air", "pressure", "precipitation",
    ],
    "tide_hourly": [
        "timestamp", "station_id", "zona_id", "lat", "lon", "source",
        "sea_level", "tide_phase", "daily_tidal_range",
    ],
    "ocean_physics": [
        "timestamp", "zona_id", "lat", "lon", "source",
        "current_speed", "sea_surface_temperature", "sea_surface_salinity",
    ],
    "ocean_validation": [
        "timestamp", "station_id", "zona_id", "source",
        "hs", "tp", "wave_direction", "validation_role",
    ],
    "bathymetry_features": [
        "zona_id", "depth_100m", "depth_500m", "depth_1km", "depth_2km",
        "mean_depth_1km", "slope_0_500m", "slope_500m_2km",
        "distance_to_10m_isobath", "distance_to_20m_isobath", "bathymetry_roughness",
    ],
    "forecast_gfs": [
        "run_time", "target_time", "horizon_hours", "zona_id", "lat", "lon", "source",
        "wind_speed", "wind_direction", "pressure", "temperature",
        "hs_forecast", "tp_forecast", "wave_direction_forecast",
    ],
}

KEY_COLUMNS = {
    "beach_geography": ["zona_id"],
    "bathymetry_features": ["zona_id"],
    "ocean_hourly": ["source", "timestamp", "zona_id"],
    "meteo_hourly": ["source", "timestamp", "zona_id"],
    "tide_hourly": ["source", "timestamp", "station_id"],
    "ocean_physics": ["source", "timestamp", "zona_id"],
    "ocean_validation": ["source", "timestamp", "station_id"],
    "forecast_gfs": ["source", "run_time", "target_time", "horizon_hours", "zona_id"],
}

TIME_COLUMNS = {
    "ocean_hourly": "timestamp",
    "meteo_hourly": "timestamp",
    "tide_hourly": "timestamp",
    "ocean_physics": "timestamp",
    "ocean_validation": "timestamp",
    "forecast_gfs": "target_time",
}

## Celda 5 — Funciones de lectura e inventario

In [6]:
def dataset_for_path(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def source_path_exists(base_path, source):
    return (Path(base_path) / f"source={source}").exists()


def read_single_parquet_schema(path):
    path = Path(path)
    if not path.exists():
        return [], None

    pf = pq.ParquetFile(path)
    schema_names = pf.schema_arrow.names
    return schema_names, pf.metadata.num_rows


def read_source_schema_and_count(base_path, source):
    base_path = Path(base_path)

    if not base_path.exists():
        return [], 0, False

    if not source_path_exists(base_path, source):
        return [], 0, False

    try:
        dataset = dataset_for_path(base_path)
        filter_expr = ds.field("source") == source
        count = dataset.count_rows(filter=filter_expr)
        schema_names = dataset.schema.names
        return schema_names, int(count), True
    except Exception as e:
        print(f"ERROR leyendo dataset {base_path} source={source}: {repr(e)}")
        return [], 0, True


def get_sample(path, kind, source=None, n=5):
    path = Path(path)

    try:
        if kind == "single_parquet":
            if not path.exists():
                return pd.DataFrame()
            return pd.read_parquet(path).head(n)

        if kind == "partitioned_source":
            if not path.exists() or not source_path_exists(path, source):
                return pd.DataFrame()

            dataset = dataset_for_path(path)
            table = dataset.head(n, filter=(ds.field("source") == source))
            return table.to_pandas()

    except Exception as e:
        print(f"No se pudo extraer sample de {path} source={source}: {repr(e)}")
        return pd.DataFrame()

    return pd.DataFrame()


def maybe_read_full(path, kind, source=None, row_count=None):
    if row_count is not None and row_count > MAX_FULL_READ_ROWS:
        return None

    path = Path(path)

    try:
        if kind == "single_parquet":
            return pd.read_parquet(path)

        if kind == "partitioned_source":
            dataset = dataset_for_path(path)
            table = dataset.to_table(filter=(ds.field("source") == source))
            return table.to_pandas()

    except Exception as e:
        print(f"No se pudo leer completo {path} source={source}: {repr(e)}")
        return None

    return None


def status_from_bool(ok, fail_message, warn_message=None, warn=False):
    if ok and not warn:
        return "PASS", ""
    if ok and warn:
        return "WARN", warn_message or ""
    return "FAIL", fail_message


def pct_missing_from_sample(sample):
    if sample.empty:
        return pd.DataFrame()

    return (
        sample.isna()
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={"index": "column", 0: "sample_missing_pct"})
    )

## Celda 6 — Inventario físico de tablas Silver

In [7]:
# Inventario rápido de tablas Silver
# Evita pyarrow.dataset.count_rows() en datasets gigantes sobre Google Drive.

FAST_EXACT_ROW_COUNT_MAX_FILES = 300

def extract_partition_values_from_path(parquet_path):
    values = {}
    for part in Path(parquet_path).parts:
        if "=" in part:
            k, v = part.split("=", 1)
            values[k] = v
    return values


def list_parquet_files_fast(path):
    path = Path(path)
    if not path.exists():
        return []
    return sorted(path.rglob("*.parquet"))


def read_schema_from_first_parquet(parquet_files):
    if not parquet_files:
        return []

    try:
        pf = pq.ParquetFile(parquet_files[0])
        schema_names = list(pf.schema_arrow.names)

        partition_values = extract_partition_values_from_path(parquet_files[0])
        for k in partition_values:
            if k not in schema_names:
                schema_names.append(k)

        return schema_names

    except Exception as e:
        print("No se pudo leer schema:", parquet_files[0], repr(e))
        return []


def fast_row_count_from_metadata(parquet_files):
    """
    Cuenta filas usando metadata solo si hay pocos ficheros.
    Si hay demasiados, devuelve un valor seguro > MAX_FULL_READ_ROWS
    para evitar lecturas completas posteriores.
    """
    if not parquet_files:
        return 0, "no_files", 0

    file_count = len(parquet_files)

    if file_count > FAST_EXACT_ROW_COUNT_MAX_FILES:
        return MAX_FULL_READ_ROWS + 1, f"skipped_exact_count_too_many_files_{file_count}", file_count

    total = 0
    errors = 0

    for p in parquet_files:
        try:
            total += pq.ParquetFile(p).metadata.num_rows
        except Exception:
            errors += 1

    note = "exact_from_parquet_metadata"
    if errors:
        note += f"_with_{errors}_metadata_errors"

    return int(total), note, file_count


def fast_sample_from_parquet_files(parquet_files, n=5):
    if not parquet_files:
        return pd.DataFrame()

    try:
        table = pq.read_table(parquet_files[0])
        sample = table.to_pandas().head(n)

        partition_values = extract_partition_values_from_path(parquet_files[0])
        for k, v in partition_values.items():
            if k not in sample.columns:
                sample[k] = v

        return sample

    except Exception as e:
        print("No se pudo leer sample:", parquet_files[0], repr(e))
        return pd.DataFrame()


inventory_rows = []
schema_rows = []
sample_missing_rows = []

for spec in tqdm(EXPECTED_OUTPUTS, desc="Inventariando Silver rápido"):
    table = spec["table"]
    source = spec["source"]
    path = Path(spec["path"])
    kind = spec["kind"]

    if kind == "single_parquet":
        exists = path.exists()
        parquet_files = [path] if exists else []
        schema_names = read_schema_from_first_parquet(parquet_files)

        if exists:
            try:
                row_count = int(pq.ParquetFile(path).metadata.num_rows)
                row_count_note = "exact_single_parquet_metadata"
                file_count = 1
            except Exception as e:
                row_count = 0
                row_count_note = f"metadata_error_{repr(e)}"
                file_count = 1
        else:
            row_count = 0
            row_count_note = "missing"
            file_count = 0

        physical_path = str(path)
        sample = fast_sample_from_parquet_files(parquet_files, n=5)

    else:
        source_dir = path / f"source={source}"
        exists = source_dir.exists()

        parquet_files = list_parquet_files_fast(source_dir) if exists else []
        schema_names = read_schema_from_first_parquet(parquet_files)

        row_count, row_count_note, file_count = fast_row_count_from_metadata(parquet_files)
        physical_path = str(source_dir)
        sample = fast_sample_from_parquet_files(parquet_files, n=5)

    rows_ok = row_count >= spec["min_rows"] or (file_count > 0 and row_count == MAX_FULL_READ_ROWS + 1)

    if not exists:
        status = "FAIL" if spec["required"] else "WARN"
        issue = "output_missing"
    elif not rows_ok:
        status = "FAIL"
        issue = f"row_count_below_min_{spec['min_rows']}"
    else:
        status = "PASS"
        issue = ""

    inventory_rows.append(
        {
            "table": table,
            "source": source,
            "kind": kind,
            "path": physical_path,
            "exists": exists,
            "row_count": row_count,
            "row_count_note": row_count_note,
            "parquet_file_count": file_count,
            "required": spec["required"],
            "min_rows": spec["min_rows"],
            "status": status,
            "issue": issue,
            "expected_role": spec["expected_role"],
            "notes": spec["notes"],
            "sample_rows": len(sample),
        }
    )

    required_columns = REQUIRED_COLUMNS.get(table, [])
    missing_columns = [c for c in required_columns if c not in schema_names]

    schema_status = "PASS" if len(missing_columns) == 0 else "FAIL"

    schema_rows.append(
        {
            "table": table,
            "source": source,
            "path": physical_path,
            "columns_count": len(schema_names),
            "required_columns_count": len(required_columns),
            "missing_required_columns_count": len(missing_columns),
            "missing_required_columns": json.dumps(missing_columns, ensure_ascii=False),
            "available_columns": json.dumps(schema_names, ensure_ascii=False),
            "status": schema_status,
        }
    )

    sm = pct_missing_from_sample(sample)

    if len(sm):
        sm["table"] = table
        sm["source"] = source
        sample_missing_rows.append(sm)

silver_table_inventory = pd.DataFrame(inventory_rows)
silver_schema_audit = pd.DataFrame(schema_rows)
silver_sample_missing = pd.concat(sample_missing_rows, ignore_index=True) if sample_missing_rows else pd.DataFrame()

display(silver_table_inventory)
display(silver_schema_audit)

silver_table_inventory.to_csv(QC_DIR / "silver_table_inventory.csv", index=False)
silver_schema_audit.to_csv(QC_DIR / "silver_schema_audit.csv", index=False)
silver_sample_missing.to_csv(QC_DIR / "silver_sample_missing_audit.csv", index=False)

print("Inventario rápido completado.")

Inventariando Silver rápido:   0%|          | 0/13 [00:00<?, ?it/s]

,table,source,kind,path,exists,row_count,row_count_note,parquet_file_count,required,min_rows,status,issue,expected_role,notes,sample_rows
0,beach_geography,DIM_ZONE,single_parquet,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,561,exact_single_parquet_metadata,1,True,1,PASS,,static_dimension,Dimensión geográfica base. Una fila por zona/p...,5
1,ocean_hourly,SIMAR,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,2273996,exact_from_parquet_metadata,188,True,1,PASS,,historical_model_wave,Oleaje histórico/modelado de SIMAR.,5
2,ocean_hourly,COPERNICUS_MARINE_WAVES,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,65472165,exact_from_parquet_metadata,234,True,1,PASS,,historical_model_wave_interpolated,Oleaje Copernicus 3h interpolado a 1h con flag=3.,5
3,meteo_hourly,SIMAR,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,2045718,exact_from_parquet_metadata,185,True,1,PASS,,historical_model_wind,Viento SIMAR.,5
4,meteo_hourly,ERA5,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,1000001,skipped_exact_count_too_many_files_2400,2400,True,1,PASS,,historical_reanalysis_precipitation,ERA5 real disponible principalmente como preci...,5
5,meteo_hourly,ERA5_PROXY,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,1000001,skipped_exact_count_too_many_files_9387,9387,True,1,PASS,,proxy_meteorology,Variables meteorológicas sintéticas/proxy. No ...,5
6,meteo_hourly,AEMET,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,27578,exact_from_parquet_metadata,65,True,1,PASS,,observed_daily_meteorology,Datos diarios AEMET guardados en tabla común c...,5
7,tide_hourly,REDMAR,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,926808,exact_from_parquet_metadata,109,True,1,PASS,,observed_sea_level,Nivel del mar y mareas observadas REDMAR.,5
8,ocean_physics,SIMAR,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,1212066,exact_from_parquet_metadata,99,True,1,PASS,,historical_model_ocean_physics,"Corrientes, temperatura y salinidad desde SIMAR.",5
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,partitioned_source,/content/drive/MyDrive/AI Projects/DeepWave Ca...,True,18682026,exact_from_parquet_metadata,225,True,1,PASS,,historical_model_ocean_physics_partial,Copernicus physics parcial: temperatura/salini...,5


,table,source,path,columns_count,required_columns_count,missing_required_columns_count,missing_required_columns,available_columns,status
0,beach_geography,DIM_ZONE,/content/drive/MyDrive/AI Projects/DeepWave Ca...,17,8,0,[],"[""zona_id"", ""nombre_zona"", ""isla"", ""municipio""...",PASS
1,ocean_hourly,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,31,8,0,[],"[""timestamp"", ""zona_id"", ""simar_point_id"", ""la...",PASS
2,ocean_hourly,COPERNICUS_MARINE_WAVES,/content/drive/MyDrive/AI Projects/DeepWave Ca...,33,8,0,[],"[""timestamp"", ""zona_id"", ""copernicus_point_id""...",PASS
3,meteo_hourly,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,23,10,0,[],"[""timestamp"", ""station_id"", ""zona_id"", ""simar_...",PASS
4,meteo_hourly,ERA5,/content/drive/MyDrive/AI Projects/DeepWave Ca...,28,10,0,[],"[""timestamp"", ""station_id"", ""zona_id"", ""era5_p...",PASS
5,meteo_hourly,ERA5_PROXY,/content/drive/MyDrive/AI Projects/DeepWave Ca...,29,10,0,[],"[""timestamp"", ""station_id"", ""zona_id"", ""era5_p...",PASS
6,meteo_hourly,AEMET,/content/drive/MyDrive/AI Projects/DeepWave Ca...,43,10,0,[],"[""timestamp"", ""date"", ""station_id"", ""station_n...",PASS
7,tide_hourly,REDMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,28,9,0,[],"[""timestamp"", ""station_id"", ""station_name"", ""z...",PASS
8,ocean_physics,SIMAR,/content/drive/MyDrive/AI Projects/DeepWave Ca...,22,8,0,[],"[""timestamp"", ""zona_id"", ""simar_point_id"", ""la...",PASS
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,/content/drive/MyDrive/AI Projects/DeepWave Ca...,22,8,0,[],"[""timestamp"", ""zona_id"", ""copernicus_point_id""...",PASS


Inventario rápido completado.


## Celda 7 — Consolidar reportes `quality_*.csv` existentes

In [8]:
quality_report_files = sorted(QC_DIR.glob("quality_*.csv"))

report_inventory = []

for p in quality_report_files:
    try:
        df = pd.read_csv(p)
        report_inventory.append(
            {
                "filename": p.name,
                "path": str(p),
                "rows": len(df),
                "columns": json.dumps(df.columns.tolist(), ensure_ascii=False),
                "read_ok": True,
                "error": "",
            }
        )
    except Exception as e:
        report_inventory.append(
            {
                "filename": p.name,
                "path": str(p),
                "rows": np.nan,
                "columns": "[]",
                "read_ok": False,
                "error": repr(e),
            }
        )

quality_report_inventory = pd.DataFrame(report_inventory)

print("Reportes quality_*.csv encontrados:", len(quality_report_inventory))
display(quality_report_inventory)

quality_report_inventory.to_csv(QC_DIR / "silver_quality_report_inventory.csv", index=False)

Reportes quality_*.csv encontrados: 53


,filename,path,rows,columns,read_ok,error
0,quality_aemet_file_summary.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,162.0,"[""filename"", ""station_id"", ""rows"", ""status"", ""...",True,
1,quality_aemet_gaps_by_station.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,8.0,"[""station_id"", ""timestamp_min"", ""timestamp_max...",True,
2,quality_aemet_global_summary.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,1.0,"[""table"", ""source"", ""rows"", ""stations"", ""times...",True,
3,quality_aemet_meteo_summary.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,1.0,"[""table"", ""source"", ""temporal_resolution"", ""ro...",True,
4,quality_aemet_missing_by_column.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,45.0,"[""column"", ""missing_pct""]",True,
5,quality_aemet_read_errors.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,NaN,[],False,EmptyDataError('No columns to parse from file')
6,quality_bathymetry_feature_errors.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,NaN,[],False,EmptyDataError('No columns to parse from file')
7,quality_bathymetry_features_missing_by_column.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,36.0,"[""column"", ""missing_pct""]",True,
8,quality_bathymetry_features_stats.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,10.0,"[""feature"", ""count"", ""mean"", ""std"", ""min"", ""5%...",True,
9,quality_bathymetry_features_summary.csv,/content/drive/MyDrive/AI Projects/DeepWave Ca...,1.0,"[""table"", ""source"", ""rows"", ""unique_zona_id"", ...",True,


## Celda 8 — Crear `dim_source_quality`

In [9]:
DIM_SOURCE_QUALITY = [
    {
        "table": "ocean_hourly",
        "source": "SIMAR",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "high",
        "notes": "Modelo histórico SIMAR de Puertos del Estado. Fuente principal de oleaje histórico.",
    },
    {
        "table": "ocean_hourly",
        "source": "COPERNICUS_MARINE_WAVES",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": True,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "high_with_interpolation",
        "notes": "Oleaje Copernicus 3h interpolado a 1h. Valores interpolados con flag=3.",
    },
    {
        "table": "meteo_hourly",
        "source": "SIMAR",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "high",
        "notes": "Viento SIMAR.",
    },
    {
        "table": "meteo_hourly",
        "source": "ERA5",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "medium_partial",
        "notes": "ERA5 disponible principalmente como precipitation/tp en esta descarga.",
    },
    {
        "table": "meteo_hourly",
        "source": "ERA5_PROXY",
        "is_real_observed": False,
        "is_model_reanalysis": False,
        "is_proxy": True,
        "is_interpolated": False,
        "use_for_training": False,
        "use_for_validation": False,
        "trust_level": "proxy_only",
        "notes": "Variables meteorológicas sintéticas para continuidad del pipeline. No usar como verdad.",
    },
    {
        "table": "meteo_hourly",
        "source": "AEMET",
        "is_real_observed": True,
        "is_model_reanalysis": False,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": True,
        "trust_level": "high_daily",
        "notes": "Observación diaria AEMET. temporal_resolution=daily.",
    },
    {
        "table": "tide_hourly",
        "source": "REDMAR",
        "is_real_observed": True,
        "is_model_reanalysis": False,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": True,
        "trust_level": "high",
        "notes": "Mareógrafos observados. Nivel del mar, marea astronómica y meteorológica.",
    },
    {
        "table": "ocean_physics",
        "source": "SIMAR",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "high",
        "notes": "Corrientes, temperatura y salinidad SIMAR.",
    },
    {
        "table": "ocean_physics",
        "source": "COPERNICUS_MARINE_PHYSICS",
        "is_real_observed": False,
        "is_model_reanalysis": True,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "medium_partial",
        "notes": "Descarga parcial: temperatura/salinidad disponibles; corrientes y zos no disponibles.",
    },
    {
        "table": "ocean_validation",
        "source": "REDEXT_REDCOS",
        "is_real_observed": True,
        "is_model_reanalysis": False,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": False,
        "use_for_validation": True,
        "trust_level": "high_temporal_low_spatial_until_coords",
        "notes": "Boyas observadas para validación. Actualmente sin coordenadas: validación temporal/global.",
    },
    {
        "table": "forecast_gfs",
        "source": "NOAA_GFS",
        "is_real_observed": False,
        "is_model_reanalysis": False,
        "is_proxy": True,
        "is_interpolated": False,
        "use_for_training": False,
        "use_for_validation": False,
        "trust_level": "operational_forecast_with_proxy",
        "notes": "GFS/GFS-Wave operativo. Variables originales + proxy trazado para nulos/swell/precipitación.",
    },
    {
        "table": "bathymetry_features",
        "source": "GEBCO_OR_EMODNET",
        "is_real_observed": False,
        "is_model_reanalysis": False,
        "is_proxy": False,
        "is_interpolated": False,
        "use_for_training": True,
        "use_for_validation": False,
        "trust_level": "static_feature_medium_high",
        "notes": "Features batimétricas estáticas. Algunas features cercanas dependen de orientación offshore heurística.",
    },
]

dim_source_quality = pd.DataFrame(DIM_SOURCE_QUALITY)
display(dim_source_quality)

dim_source_quality.to_csv(META_DIR / "dim_source_quality.csv", index=False)

,table,source,is_real_observed,is_model_reanalysis,is_proxy,is_interpolated,use_for_training,use_for_validation,trust_level,notes
0,ocean_hourly,SIMAR,False,True,False,False,True,False,high,Modelo histórico SIMAR de Puertos del Estado. ...
1,ocean_hourly,COPERNICUS_MARINE_WAVES,False,True,False,True,True,False,high_with_interpolation,Oleaje Copernicus 3h interpolado a 1h. Valores...
2,meteo_hourly,SIMAR,False,True,False,False,True,False,high,Viento SIMAR.
3,meteo_hourly,ERA5,False,True,False,False,True,False,medium_partial,ERA5 disponible principalmente como precipitat...
4,meteo_hourly,ERA5_PROXY,False,False,True,False,False,False,proxy_only,Variables meteorológicas sintéticas para conti...
5,meteo_hourly,AEMET,True,False,False,False,True,True,high_daily,Observación diaria AEMET. temporal_resolution=...
6,tide_hourly,REDMAR,True,False,False,False,True,True,high,"Mareógrafos observados. Nivel del mar, marea a..."
7,ocean_physics,SIMAR,False,True,False,False,True,False,high,"Corrientes, temperatura y salinidad SIMAR."
8,ocean_physics,COPERNICUS_MARINE_PHYSICS,False,True,False,False,True,False,medium_partial,Descarga parcial: temperatura/salinidad dispon...
9,ocean_validation,REDEXT_REDCOS,True,False,False,False,False,True,high_temporal_low_spatial_until_coords,Boyas observadas para validación. Actualmente ...


## Celda 9 — Salvedades conocidas y aceptadas

In [10]:
KNOWN_CAVEATS = [
    {
        "area": "ERA5",
        "severity": "WARN",
        "issue": "ERA5 descargado solo contiene tp/precipitation en los archivos disponibles.",
        "impact": "No se genera ocean_hourly ERA5 real. Meteorología completa se cubre con ERA5_PROXY, SIMAR y AEMET.",
        "action": "Documentar. Si se re-descarga ERA5 completo, reejecutar 03_silver_era5.",
    },
    {
        "area": "ERA5_PROXY",
        "severity": "WARN",
        "issue": "Variables meteorológicas proxy/sintéticas.",
        "impact": "No deben usarse como verdad observada ni para validación.",
        "action": "Filtrar o ponderar en Gold según caso de uso.",
    },
    {
        "area": "Copernicus Marine Physics",
        "severity": "WARN",
        "issue": "Descarga physics no contiene uo/vo ni zos detectables.",
        "impact": "Corrientes se cubren con SIMAR. Nivel del mar se cubre con REDMAR.",
        "action": "Documentar. Si se descarga producto con uo/vo/zos, reejecutar 05.",
    },
    {
        "area": "REDEXT/REDCOS",
        "severity": "WARN",
        "issue": "Boyas observadas sin coordenadas en los CSV.",
        "impact": "Validación espacial fina no disponible; quedan como CAN_VALIDATION_UNASSIGNED.",
        "action": "Añadir coordenadas revisadas manualmente desde Puertos del Estado si hay tiempo.",
    },
    {
        "area": "Bathymetry",
        "severity": "WARN",
        "issue": "Orientación offshore aproximada por heurística.",
        "impact": "depth_100m/depth_500m y slope_0_500m pueden ser menos fiables en algunas playas.",
        "action": "En Gold priorizar mean_depth_1km, isóbatas y roughness; mejorar con línea de costa si hay tiempo.",
    },
    {
        "area": "GFS forecast",
        "severity": "WARN",
        "issue": "Oleaje costero, swell y precipitación tienen proxy sintético trazado.",
        "impact": "forecast_gfs es válido para inferencia operativa, pero separar valores originales de sintéticos usando *_was_synthetic y flag=3.",
        "action": "En Gold/producción mantener columnas de trazabilidad.",
    },
]

known_caveats_df = pd.DataFrame(KNOWN_CAVEATS)
display(known_caveats_df)

known_caveats_df.to_csv(QC_DIR / "silver_known_caveats.csv", index=False)

,area,severity,issue,impact,action
0,ERA5,WARN,ERA5 descargado solo contiene tp/precipitation...,No se genera ocean_hourly ERA5 real. Meteorolo...,"Documentar. Si se re-descarga ERA5 completo, r..."
1,ERA5_PROXY,WARN,Variables meteorológicas proxy/sintéticas.,No deben usarse como verdad observada ni para ...,Filtrar o ponderar en Gold según caso de uso.
2,Copernicus Marine Physics,WARN,Descarga physics no contiene uo/vo ni zos dete...,Corrientes se cubren con SIMAR. Nivel del mar ...,Documentar. Si se descarga producto con uo/vo/...
3,REDEXT/REDCOS,WARN,Boyas observadas sin coordenadas en los CSV.,Validación espacial fina no disponible; quedan...,Añadir coordenadas revisadas manualmente desde...
4,Bathymetry,WARN,Orientación offshore aproximada por heurística.,depth_100m/depth_500m y slope_0_500m pueden se...,"En Gold priorizar mean_depth_1km, isóbatas y r..."
5,GFS forecast,WARN,"Oleaje costero, swell y precipitación tienen p...",forecast_gfs es válido para inferencia operati...,En Gold/producción mantener columnas de trazab...


## Celda 10 — Auditoría de claves y duplicados en tablas pequeñas/medianas

In [11]:
duplicate_audit_rows = []

for spec in tqdm(EXPECTED_OUTPUTS, desc="Auditando duplicados cuando sea seguro"):
    table = spec["table"]
    source = spec["source"]
    path = Path(spec["path"])
    kind = spec["kind"]

    inv = silver_table_inventory[
        (silver_table_inventory["table"] == table)
        & (silver_table_inventory["source"] == source)
    ]

    if inv.empty:
        continue

    row_count = int(inv["row_count"].iloc[0])
    exists = bool(inv["exists"].iloc[0])

    key_cols = KEY_COLUMNS.get(table, [])

    if not exists:
        duplicate_audit_rows.append(
            {
                "table": table,
                "source": source,
                "row_count": row_count,
                "key_columns": json.dumps(key_cols),
                "checked": False,
                "reason": "missing_output",
                "duplicates_count": np.nan,
                "status": "FAIL" if spec["required"] else "WARN",
            }
        )
        continue

    if row_count > MAX_FULL_READ_ROWS:
        duplicate_audit_rows.append(
            {
                "table": table,
                "source": source,
                "row_count": row_count,
                "key_columns": json.dumps(key_cols),
                "checked": False,
                "reason": f"too_large_over_{MAX_FULL_READ_ROWS}_rows",
                "duplicates_count": np.nan,
                "status": "WARN",
            }
        )
        continue

    df_full = maybe_read_full(path, kind, source=source if kind == "partitioned_source" else None, row_count=row_count)

    if df_full is None or df_full.empty:
        duplicate_audit_rows.append(
            {
                "table": table,
                "source": source,
                "row_count": row_count,
                "key_columns": json.dumps(key_cols),
                "checked": False,
                "reason": "could_not_read_full_or_empty",
                "duplicates_count": np.nan,
                "status": "WARN",
            }
        )
        continue

    missing_keys = [c for c in key_cols if c not in df_full.columns]

    if missing_keys:
        duplicate_audit_rows.append(
            {
                "table": table,
                "source": source,
                "row_count": row_count,
                "key_columns": json.dumps(key_cols),
                "checked": False,
                "reason": "missing_key_columns_" + ",".join(missing_keys),
                "duplicates_count": np.nan,
                "status": "FAIL",
            }
        )
        continue

    duplicates_count = int(df_full.duplicated(subset=key_cols).sum())
    status = "PASS" if duplicates_count == 0 else "FAIL"

    duplicate_audit_rows.append(
        {
            "table": table,
            "source": source,
            "row_count": row_count,
            "key_columns": json.dumps(key_cols),
            "checked": True,
            "reason": "",
            "duplicates_count": duplicates_count,
            "status": status,
        }
    )

duplicate_audit = pd.DataFrame(duplicate_audit_rows)
display(duplicate_audit)

duplicate_audit.to_csv(QC_DIR / "silver_duplicate_audit.csv", index=False)

Auditando duplicados cuando sea seguro:   0%|          | 0/13 [00:00<?, ?it/s]

,table,source,row_count,key_columns,checked,reason,duplicates_count,status
0,beach_geography,DIM_ZONE,561,"[""zona_id""]",True,,0.0,PASS
1,ocean_hourly,SIMAR,2273996,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
2,ocean_hourly,COPERNICUS_MARINE_WAVES,65472165,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
3,meteo_hourly,SIMAR,2045718,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
4,meteo_hourly,ERA5,1000001,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
5,meteo_hourly,ERA5_PROXY,1000001,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
6,meteo_hourly,AEMET,27578,"[""source"", ""timestamp"", ""zona_id""]",True,,0.0,PASS
7,tide_hourly,REDMAR,926808,"[""source"", ""timestamp"", ""station_id""]",True,,0.0,PASS
8,ocean_physics,SIMAR,1212066,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,18682026,"[""source"", ""timestamp"", ""zona_id""]",False,too_large_over_1000000_rows,NaN,WARN


## Celda 11 — Auditoría de integridad de `zona_id` en tablas manejables

In [12]:
zone_integrity_rows = []

beach_path = SILVER_DIR / "beach_geography" / "beach_geography.parquet"

if beach_path.exists():
    dim_zone = pd.read_parquet(beach_path)
    known_zones = set(dim_zone["zona_id"].dropna().astype(str))
else:
    known_zones = set()

allowed_external_zones = {"CAN_VALIDATION_UNASSIGNED"}

for spec in tqdm(EXPECTED_OUTPUTS, desc="Auditando zona_id cuando sea seguro"):
    table = spec["table"]
    source = spec["source"]
    path = Path(spec["path"])
    kind = spec["kind"]

    if table in ["beach_geography"]:
        continue

    inv = silver_table_inventory[
        (silver_table_inventory["table"] == table)
        & (silver_table_inventory["source"] == source)
    ]

    if inv.empty:
        continue

    row_count = int(inv["row_count"].iloc[0])
    exists = bool(inv["exists"].iloc[0])

    if not exists:
        continue

    if row_count > MAX_FULL_READ_ROWS:
        sample = get_sample(path, kind, source=source if kind == "partitioned_source" else None, n=5000)
        check_mode = "sample_5000"
    else:
        sample = maybe_read_full(path, kind, source=source if kind == "partitioned_source" else None, row_count=row_count)
        check_mode = "full"

    if sample is None or sample.empty or "zona_id" not in sample.columns:
        zone_integrity_rows.append(
            {
                "table": table,
                "source": source,
                "check_mode": check_mode,
                "rows_checked": 0,
                "unknown_zone_count": np.nan,
                "unknown_zone_examples": "[]",
                "status": "WARN",
                "notes": "No se pudo comprobar zona_id.",
            }
        )
        continue

    z = sample["zona_id"].dropna().astype(str)
    unknown = sorted(set(z) - known_zones - allowed_external_zones)

    status = "PASS" if len(unknown) == 0 else "FAIL"

    zone_integrity_rows.append(
        {
            "table": table,
            "source": source,
            "check_mode": check_mode,
            "rows_checked": len(sample),
            "unknown_zone_count": len(unknown),
            "unknown_zone_examples": json.dumps(unknown[:20], ensure_ascii=False),
            "status": status,
            "notes": "CAN_VALIDATION_UNASSIGNED permitido para REDEXT/REDCOS sin coordenadas.",
        }
    )

zone_integrity_audit = pd.DataFrame(zone_integrity_rows)
display(zone_integrity_audit)

zone_integrity_audit.to_csv(QC_DIR / "silver_zone_integrity_audit.csv", index=False)

Auditando zona_id cuando sea seguro:   0%|          | 0/13 [00:00<?, ?it/s]

,table,source,check_mode,rows_checked,unknown_zone_count,unknown_zone_examples,status,notes
0,ocean_hourly,SIMAR,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
1,ocean_hourly,COPERNICUS_MARINE_WAVES,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
2,meteo_hourly,SIMAR,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
3,meteo_hourly,ERA5,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
4,meteo_hourly,ERA5_PROXY,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
5,meteo_hourly,AEMET,full,27578,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
6,tide_hourly,REDMAR,full,926808,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
7,ocean_physics,SIMAR,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
8,ocean_physics,COPERNICUS_MARINE_PHYSICS,sample_5000,5000,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...
9,ocean_validation,REDEXT_REDCOS,full,730445,0,[],PASS,CAN_VALIDATION_UNASSIGNED permitido para REDEX...


## Celda 12 — Auditoría específica de proxies/sintéticos e interpolados

In [13]:
proxy_audit_rows = []

# Forecast GFS con proxy.
forecast_spec = next(x for x in EXPECTED_OUTPUTS if x["table"] == "forecast_gfs")
forecast_inv = silver_table_inventory[
    (silver_table_inventory["table"] == "forecast_gfs")
    & (silver_table_inventory["source"] == "NOAA_GFS")
]

if not forecast_inv.empty and forecast_inv["exists"].iloc[0]:
    forecast_count = int(forecast_inv["row_count"].iloc[0])
    forecast_df = maybe_read_full(
        forecast_spec["path"],
        forecast_spec["kind"],
        source="NOAA_GFS",
        row_count=forecast_count,
    )

    if forecast_df is not None and not forecast_df.empty:
        synthetic_cols = [c for c in forecast_df.columns if c.endswith("_was_synthetic")]

        for col in synthetic_cols:
            variable = col.replace("_was_synthetic", "")
            proxy_audit_rows.append(
                {
                    "table": "forecast_gfs",
                    "source": "NOAA_GFS",
                    "variable": variable,
                    "traceability_column": col,
                    "synthetic_pct": float(forecast_df[col].fillna(False).astype(bool).mean() * 100),
                    "missing_pct_after_proxy": float(forecast_df[variable].isna().mean() * 100) if variable in forecast_df.columns else np.nan,
                    "status": "PASS",
                    "notes": "Valores proxy trazados correctamente si synthetic_pct > 0.",
                }
            )

# Copernicus interpolated.
cop_spec = [x for x in EXPECTED_OUTPUTS if x["table"] == "ocean_hourly" and x["source"] == "COPERNICUS_MARINE_WAVES"][0]
cop_inv = silver_table_inventory[
    (silver_table_inventory["table"] == "ocean_hourly")
    & (silver_table_inventory["source"] == "COPERNICUS_MARINE_WAVES")
]

if not cop_inv.empty and cop_inv["exists"].iloc[0]:
    sample = get_sample(cop_spec["path"], "partitioned_source", source="COPERNICUS_MARINE_WAVES", n=10000)

    if "interpolated" in sample.columns:
        proxy_audit_rows.append(
            {
                "table": "ocean_hourly",
                "source": "COPERNICUS_MARINE_WAVES",
                "variable": "interpolated",
                "traceability_column": "interpolated",
                "synthetic_pct": float(sample["interpolated"].fillna(False).astype(bool).mean() * 100),
                "missing_pct_after_proxy": np.nan,
                "status": "PASS",
                "notes": "Muestra revisada. El reporte fuente indicaba ~66.66% por interpolación 3h→1h.",
            }
        )

proxy_traceability_audit = pd.DataFrame(proxy_audit_rows)
display(proxy_traceability_audit)

proxy_traceability_audit.to_csv(QC_DIR / "silver_proxy_traceability_audit.csv", index=False)

,table,source,variable,traceability_column,synthetic_pct,missing_pct_after_proxy,status,notes
0,forecast_gfs,NOAA_GFS,hs_forecast,hs_forecast_was_synthetic,40.819964,0.0,PASS,Valores proxy trazados correctamente si synthe...
1,forecast_gfs,NOAA_GFS,tp_forecast,tp_forecast_was_synthetic,40.819964,0.0,PASS,Valores proxy trazados correctamente si synthe...
2,forecast_gfs,NOAA_GFS,wave_direction_forecast,wave_direction_forecast_was_synthetic,40.819964,0.0,PASS,Valores proxy trazados correctamente si synthe...
3,forecast_gfs,NOAA_GFS,swell_forecast,swell_forecast_was_synthetic,100.000000,0.0,PASS,Valores proxy trazados correctamente si synthe...
4,forecast_gfs,NOAA_GFS,swell_period_forecast,swell_period_forecast_was_synthetic,100.000000,0.0,PASS,Valores proxy trazados correctamente si synthe...
5,forecast_gfs,NOAA_GFS,swell_direction_forecast,swell_direction_forecast_was_synthetic,41.270840,0.0,PASS,Valores proxy trazados correctamente si synthe...
6,forecast_gfs,NOAA_GFS,precipitation_rate_mm_h,precipitation_rate_mm_h_was_synthetic,100.000000,0.0,PASS,Valores proxy trazados correctamente si synthe...
7,ocean_hourly,COPERNICUS_MARINE_WAVES,interpolated,interpolated,66.660000,NaN,PASS,Muestra revisada. El reporte fuente indicaba ~...


## Celda 13 — Resumen global de estado

In [14]:
audit_components = []

def append_component(component_name, df, status_col="status"):
    if df is None or df.empty or status_col not in df.columns:
        audit_components.append(
            {
                "component": component_name,
                "pass_count": 0,
                "warn_count": 1,
                "fail_count": 0,
                "status": "WARN",
                "notes": "Componente vacío o sin columna status.",
            }
        )
        return

    pass_count = int((df[status_col] == "PASS").sum())
    warn_count = int((df[status_col] == "WARN").sum())
    fail_count = int((df[status_col] == "FAIL").sum())

    if fail_count > 0:
        status = "FAIL"
    elif warn_count > 0:
        status = "WARN"
    else:
        status = "PASS"

    audit_components.append(
        {
            "component": component_name,
            "pass_count": pass_count,
            "warn_count": warn_count,
            "fail_count": fail_count,
            "status": status,
            "notes": "",
        }
    )

append_component("table_inventory", silver_table_inventory)
append_component("schema_audit", silver_schema_audit)
append_component("duplicate_audit", duplicate_audit)
append_component("zone_integrity_audit", zone_integrity_audit)
append_component("proxy_traceability_audit", proxy_traceability_audit)

silver_global_audit_summary = pd.DataFrame(audit_components)

overall_fail_count = int(silver_global_audit_summary["fail_count"].sum())
overall_warn_count = int(silver_global_audit_summary["warn_count"].sum())

if overall_fail_count > 0:
    overall_status = "FAIL"
elif overall_warn_count > 0:
    overall_status = "WARN"
else:
    overall_status = "PASS"

overall_row = pd.DataFrame(
    [
        {
            "component": "OVERALL",
            "pass_count": int(silver_global_audit_summary["pass_count"].sum()),
            "warn_count": overall_warn_count,
            "fail_count": overall_fail_count,
            "status": overall_status,
            "notes": "WARN puede ser aceptable si coincide con salvedades conocidas.",
        }
    ]
)

silver_global_audit_summary = pd.concat([silver_global_audit_summary, overall_row], ignore_index=True)

display(silver_global_audit_summary)

silver_global_audit_summary.to_csv(QC_DIR / "silver_global_audit_summary.csv", index=False)

,component,pass_count,warn_count,fail_count,status,notes
0,table_inventory,13,0,0,PASS,
1,schema_audit,13,0,0,PASS,
2,duplicate_audit,6,7,0,WARN,
3,zone_integrity_audit,12,0,0,PASS,
4,proxy_traceability_audit,8,0,0,PASS,
5,OVERALL,52,7,0,WARN,WARN puede ser aceptable si coincide con salve...


## Celda 14 — Estado por fuente listo para decisión Silver → Gold

In [15]:
sources_status_rows = []

for _, inv in silver_table_inventory.iterrows():
    table = inv["table"]
    source = inv["source"]

    schema_status = silver_schema_audit[
        (silver_schema_audit["table"] == table)
        & (silver_schema_audit["source"] == source)
    ]["status"]

    dup_status = duplicate_audit[
        (duplicate_audit["table"] == table)
        & (duplicate_audit["source"] == source)
    ]["status"]

    zone_status = zone_integrity_audit[
        (zone_integrity_audit["table"] == table)
        & (zone_integrity_audit["source"] == source)
    ]["status"]

    statuses = [inv["status"]]

    if len(schema_status):
        statuses.append(schema_status.iloc[0])

    if len(dup_status):
        statuses.append(dup_status.iloc[0])

    if len(zone_status):
        statuses.append(zone_status.iloc[0])

    if "FAIL" in statuses:
        final_status = "FAIL"
    elif "WARN" in statuses:
        final_status = "WARN"
    else:
        final_status = "PASS"

    source_quality_match = dim_source_quality[
        (dim_source_quality["table"] == table)
        & (dim_source_quality["source"] == source)
    ]

    if len(source_quality_match):
        sq = source_quality_match.iloc[0].to_dict()
    else:
        sq = {}

    sources_status_rows.append(
        {
            "table": table,
            "source": source,
            "rows": inv["row_count"],
            "status": final_status,
            "inventory_status": inv["status"],
            "schema_status": schema_status.iloc[0] if len(schema_status) else "NA",
            "duplicate_status": dup_status.iloc[0] if len(dup_status) else "NA",
            "zone_status": zone_status.iloc[0] if len(zone_status) else "NA",
            "expected_role": inv["expected_role"],
            "use_for_training": sq.get("use_for_training", np.nan),
            "use_for_validation": sq.get("use_for_validation", np.nan),
            "trust_level": sq.get("trust_level", ""),
            "notes": inv["notes"],
        }
    )

silver_sources_status = pd.DataFrame(sources_status_rows)
display(silver_sources_status)

silver_sources_status.to_csv(QC_DIR / "silver_sources_status.csv", index=False)

,table,source,rows,status,inventory_status,schema_status,duplicate_status,zone_status,expected_role,use_for_training,use_for_validation,trust_level,notes
0,beach_geography,DIM_ZONE,561,PASS,PASS,PASS,PASS,NA,static_dimension,NaN,NaN,,Dimensión geográfica base. Una fila por zona/p...
1,ocean_hourly,SIMAR,2273996,WARN,PASS,PASS,WARN,PASS,historical_model_wave,True,False,high,Oleaje histórico/modelado de SIMAR.
2,ocean_hourly,COPERNICUS_MARINE_WAVES,65472165,WARN,PASS,PASS,WARN,PASS,historical_model_wave_interpolated,True,False,high_with_interpolation,Oleaje Copernicus 3h interpolado a 1h con flag=3.
3,meteo_hourly,SIMAR,2045718,WARN,PASS,PASS,WARN,PASS,historical_model_wind,True,False,high,Viento SIMAR.
4,meteo_hourly,ERA5,1000001,WARN,PASS,PASS,WARN,PASS,historical_reanalysis_precipitation,True,False,medium_partial,ERA5 real disponible principalmente como preci...
5,meteo_hourly,ERA5_PROXY,1000001,WARN,PASS,PASS,WARN,PASS,proxy_meteorology,False,False,proxy_only,Variables meteorológicas sintéticas/proxy. No ...
6,meteo_hourly,AEMET,27578,PASS,PASS,PASS,PASS,PASS,observed_daily_meteorology,True,True,high_daily,Datos diarios AEMET guardados en tabla común c...
7,tide_hourly,REDMAR,926808,PASS,PASS,PASS,PASS,PASS,observed_sea_level,True,True,high,Nivel del mar y mareas observadas REDMAR.
8,ocean_physics,SIMAR,1212066,WARN,PASS,PASS,WARN,PASS,historical_model_ocean_physics,True,False,high,"Corrientes, temperatura y salinidad desde SIMAR."
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,18682026,WARN,PASS,PASS,WARN,PASS,historical_model_ocean_physics_partial,True,False,medium_partial,Copernicus physics parcial: temperatura/salini...


## Celda 15 — Decisión final automática

In [16]:
print("=== RESULTADO AUDITORÍA SILVER ===")
display(silver_global_audit_summary)
display(silver_sources_status)

critical_fails = silver_sources_status[silver_sources_status["status"] == "FAIL"].copy()

# Duplicados no chequeados en tablas gigantes quedan WARN, no FAIL.
print("\nFuentes con FAIL:")
display(critical_fails)

print("\nSalvedades conocidas:")
display(known_caveats_df)

print("\nArchivos generados:")
for p in [
    QC_DIR / "silver_global_audit_summary.csv",
    QC_DIR / "silver_sources_status.csv",
    QC_DIR / "silver_table_inventory.csv",
    QC_DIR / "silver_schema_audit.csv",
    QC_DIR / "silver_duplicate_audit.csv",
    QC_DIR / "silver_zone_integrity_audit.csv",
    QC_DIR / "silver_proxy_traceability_audit.csv",
    QC_DIR / "silver_known_caveats.csv",
    META_DIR / "dim_source_quality.csv",
]:
    print("-", p)

if len(critical_fails) > 0:
    msg = (
        "La auditoría Silver encontró FAIL críticos. "
        "Revisa silver_sources_status.csv antes de pasar a Gold."
    )

    if FAIL_ON_CRITICAL:
        raise ValueError(msg)
    else:
        print("AVISO:", msg)
else:
    print("\n✅ Auditoría Silver sin FAIL críticos.")
    print("La capa Silver está lista para pasar a Gold, manteniendo documentadas las salvedades WARN.")

=== RESULTADO AUDITORÍA SILVER ===


,component,pass_count,warn_count,fail_count,status,notes
0,table_inventory,13,0,0,PASS,
1,schema_audit,13,0,0,PASS,
2,duplicate_audit,6,7,0,WARN,
3,zone_integrity_audit,12,0,0,PASS,
4,proxy_traceability_audit,8,0,0,PASS,
5,OVERALL,52,7,0,WARN,WARN puede ser aceptable si coincide con salve...


,table,source,rows,status,inventory_status,schema_status,duplicate_status,zone_status,expected_role,use_for_training,use_for_validation,trust_level,notes
0,beach_geography,DIM_ZONE,561,PASS,PASS,PASS,PASS,NA,static_dimension,NaN,NaN,,Dimensión geográfica base. Una fila por zona/p...
1,ocean_hourly,SIMAR,2273996,WARN,PASS,PASS,WARN,PASS,historical_model_wave,True,False,high,Oleaje histórico/modelado de SIMAR.
2,ocean_hourly,COPERNICUS_MARINE_WAVES,65472165,WARN,PASS,PASS,WARN,PASS,historical_model_wave_interpolated,True,False,high_with_interpolation,Oleaje Copernicus 3h interpolado a 1h con flag=3.
3,meteo_hourly,SIMAR,2045718,WARN,PASS,PASS,WARN,PASS,historical_model_wind,True,False,high,Viento SIMAR.
4,meteo_hourly,ERA5,1000001,WARN,PASS,PASS,WARN,PASS,historical_reanalysis_precipitation,True,False,medium_partial,ERA5 real disponible principalmente como preci...
5,meteo_hourly,ERA5_PROXY,1000001,WARN,PASS,PASS,WARN,PASS,proxy_meteorology,False,False,proxy_only,Variables meteorológicas sintéticas/proxy. No ...
6,meteo_hourly,AEMET,27578,PASS,PASS,PASS,PASS,PASS,observed_daily_meteorology,True,True,high_daily,Datos diarios AEMET guardados en tabla común c...
7,tide_hourly,REDMAR,926808,PASS,PASS,PASS,PASS,PASS,observed_sea_level,True,True,high,Nivel del mar y mareas observadas REDMAR.
8,ocean_physics,SIMAR,1212066,WARN,PASS,PASS,WARN,PASS,historical_model_ocean_physics,True,False,high,"Corrientes, temperatura y salinidad desde SIMAR."
9,ocean_physics,COPERNICUS_MARINE_PHYSICS,18682026,WARN,PASS,PASS,WARN,PASS,historical_model_ocean_physics_partial,True,False,medium_partial,Copernicus physics parcial: temperatura/salini...



Fuentes con FAIL:


,table,source,rows,status,inventory_status,schema_status,duplicate_status,zone_status,expected_role,use_for_training,use_for_validation,trust_level,notes



Salvedades conocidas:


,area,severity,issue,impact,action
0,ERA5,WARN,ERA5 descargado solo contiene tp/precipitation...,No se genera ocean_hourly ERA5 real. Meteorolo...,"Documentar. Si se re-descarga ERA5 completo, r..."
1,ERA5_PROXY,WARN,Variables meteorológicas proxy/sintéticas.,No deben usarse como verdad observada ni para ...,Filtrar o ponderar en Gold según caso de uso.
2,Copernicus Marine Physics,WARN,Descarga physics no contiene uo/vo ni zos dete...,Corrientes se cubren con SIMAR. Nivel del mar ...,Documentar. Si se descarga producto con uo/vo/...
3,REDEXT/REDCOS,WARN,Boyas observadas sin coordenadas en los CSV.,Validación espacial fina no disponible; quedan...,Añadir coordenadas revisadas manualmente desde...
4,Bathymetry,WARN,Orientación offshore aproximada por heurística.,depth_100m/depth_500m y slope_0_500m pueden se...,"En Gold priorizar mean_depth_1km, isóbatas y r..."
5,GFS forecast,WARN,"Oleaje costero, swell y precipitación tienen p...",forecast_gfs es válido para inferencia operati...,En Gold/producción mantener columnas de trazab...



Archivos generados:
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_global_audit_summary.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_sources_status.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_table_inventory.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_schema_audit.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_duplicate_audit.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_zone_integrity_audit.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_proxy_traceability_audit.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_quality_reports/silver_known_caveats.csv
- /content/drive/MyDrive/AI Projects/DeepWave Canarias/silver/_metadata/dim_source_quality.csv

✅ Auditoría Silver sin